# Notebook 3 — Local Grounded Reader

This notebook adds the fixed `qwen3.5:9b-q4_K_M` reader to the Nomic top-five retrieval results. Generation runs locally through Ollama with temperature 0, a fixed seed, hidden reasoning disabled, and JSON output. The reader must cite supplied evidence or abstain.

In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
(ROOT / 'results' / 'figures').mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)
print('Frozen test split accessed:', False)

Project root: <PROJECT_ROOT>
Frozen test split accessed: False


In [2]:
import json
import pandas as pd
from src.baseline import evaluate, save_results
from src.local_pipeline import evaluate_reader

summary, rows = evaluate(ROOT, 'ollama', 'development', 5, with_reader=True)
summary['reader_metrics'] = evaluate_reader(rows)
save_results(ROOT / 'results', 'ollama_dense_development_k5_reader', summary, rows)
summary

{'experiment_name': 'week3_retrieval_baseline',
 'dataset_version': '1.0.0',
 'retriever': 'ollama_embedding_cosine',
 'embedding_model': 'nomic-embed-text',
 'reader_model': 'qwen3.5:9b-q4_K_M',
 'split': 'development',
 'top_k': 5,
 'question_count': 30,
 'elapsed_seconds': 73.8121,
 'average_seconds_per_question': 2.460402,
 'metrics': {'hit_rate_at_5': 1.0,
  'recall_at_5': 1.0,
  'precision_at_5': 0.3,
  'mean_reciprocal_rank': 0.8125,
  'answerable_questions': 24,
  'unanswerable_questions_excluded_from_retrieval_metrics': 6,
  'by_category': {'direct_retrieval': {'questions': 6, 'hit_rate_at_5': 1.0},
   'cross_document_synthesis': {'questions': 6, 'hit_rate_at_5': 1.0},
   'temporal_update': {'questions': 6, 'hit_rate_at_5': 1.0},
   'conflicting_evidence': {'questions': 6, 'hit_rate_at_5': 1.0},
   'unanswerable_abstention': {'questions': 6, 'hit_rate_at_5': None}}},
 'test_set_accessed': False,
 'reader_metrics': {'abstention_accuracy': 1.0,
  'citation_hit_rate_answerable': 

In [3]:
examples = []
for row in rows[:6]:
    examples.append({
        'question_id': row['question_id'],
        'gold_documents': row['gold_document_ids'],
        'retrieved_documents': row['retrieved_document_ids'],
        'answer': row['reader']['answer'],
        'citations': row['reader']['citations'],
        'abstained': row['reader']['abstained'],
        'latency_seconds': row['reader']['latency_seconds'],
    })
pd.DataFrame(examples)

,question_id,gold_documents,retrieved_documents,answer,citations,abstained,latency_seconds
0,RCD-DIR-03,[RCD-VAL-v2],"[RCD-VAL-v1, RCD-VAL-v2, RCD-CHR-v2, RCD-CHR-v...",The primary validation metric reported in the ...,[RCD-VAL-v2],False,2.1860
1,RCD-CRO-03,"[RCD-DEV-v1, RCD-DQ-2026Q2]","[RCD-DEV-v1, RCD-DQ-2026Q2, RCD-CHR-v2, RCD-CH...",The governed development data consists of anon...,"[RCD-DEV-v1, RCD-DQ-2026Q2]",False,2.8380
2,RCD-TEM-03,[RCD-ISS-2026-01],"[RCD-MRC-2026-09, RCD-ISS-2026-01, RCD-CHR-v2,...",OPEN,"[RCD-ISS-2026-01, RCD-REM-2026-01]",False,1.7344
3,RCD-CON-03,"[RCD-MON-2026Q1, RCD-MON-2026Q2]","[RCD-MON-2026Q2, RCD-MON-2026Q1, RCD-CHR-v1, R...",The Q2 result should drive the latest risk dec...,"[RCD-MON-2026Q2, RCD-MON-2026Q1]",False,3.0502
4,RCD-UNA-03,[],"[RCD-CHR-v2, RCD-CHR-v1, RCD-DEV-v1, RCD-DQ-20...",NaN,[],True,3.6760
5,CTF-DIR-03,[CTF-VAL-v2],"[CTF-VAL-v2, CTF-VAL-v1, CTF-CHR-v2, CTF-MON-2...",The primary validation metric reported in the ...,[CTF-VAL-v2],False,2.0850


In [4]:
reader_frame = pd.DataFrame([{
    'question_id': row['question_id'],
    'category': row['category'],
    'should_abstain': row['should_abstain'],
    'abstained': row['reader']['abstained'],
    'citation_hit': bool(set(row['reader']['citations']) & set(row['gold_document_ids'])) if row['gold_document_ids'] else None,
    'latency_seconds': row['reader']['latency_seconds'],
} for row in rows])
reader_frame.groupby('category').agg(questions=('question_id','count'), mean_latency=('latency_seconds','mean'))

,questions,mean_latency
category,,
conflicting_evidence,6,3.108117
cross_document_synthesis,6,3.223283
direct_retrieval,6,2.071233
temporal_update,6,1.609450
unanswerable_abstention,6,2.229233


## Interpretation

These metrics separate retrieval from generation. Citation hit rate asks whether the reader cited gold evidence for answerable questions; abstention accuracy evaluates supported versus unsupported questions; grounded-citation rate verifies that citations came from the supplied top-five documents. This is not a claim of production readiness or full semantic answer correctness.